# 프롬프트 캐싱 손익분기

캐시 쓰기, 읽기 배수로 손익분기 재사용 횟수를 계산하고, 호출 간격과 TTL에 따른 비용을 시뮬레이션합니다. API 호출 없이 접두부 비용만 계산합니다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/prompt_cache_breakeven.ipynb)

Colab에서 CPU 런타임을 선택하고 위에서부터 셀을 실행하세요. 첫 코드 셀에서 `QUICK = True`로 바꾸면 축소 조건으로 실행합니다.

In [ ]:
QUICK = False

import random
from fractions import Fraction
from math import ceil, floor

# 기본 입력 단가를 1로 둔 가정값입니다. 배수는 문자열로 바꾸세요.
W5 = Fraction("1.25")
W60 = Fraction("2.0")
R = Fraction("0.1")
ALT_R = Fraction("0.025")  # 비교 실험의 읽기 배수
MIN_CACHE_TOKENS = 1024

if min(W5, W60, R, ALT_R) < 0:
    raise ValueError("비용 배수는 0 이상이어야 합니다.")
if MIN_CACHE_TOKENS < 1:
    raise ValueError("최소 캐시 길이는 1 이상이어야 합니다.")

SEED = 20261119
TTLS = (("5분", 300, W5), ("1시간", 3600, W60))
HIT_COUNTS = (0, 1, 2, 3, 5, 10, 100)
MEAN_GAPS = (
    ("30초", 30), ("1분", 60), ("2분", 120), ("4분", 240),
    ("5분", 300), ("6분", 360), ("10분", 600), ("30분", 1800),
    ("1시간", 3600), ("2시간", 7200),
)


def cost_ratio(calls, hits, writes, write, read=R):
    ordinary_calls = calls - hits - writes
    return (ordinary_calls + write * writes + read * hits) / calls


def simulate(gaps, ttl, prefix_tokens, min_cache_tokens, refresh=True):
    """간격 목록 앞에 시각 0의 첫 호출을 두고 히트와 쓰기를 센다."""
    if prefix_tokens < min_cache_tokens:
        return 0, 0

    now = 0
    expires_at = ttl
    hits = 0
    writes = 1
    for gap in gaps:
        now += gap
        if now < expires_at:
            hits += 1
            if refresh:
                expires_at = now + ttl
        else:
            writes += 1
            expires_at = now + ttl
    return hits, writes


def format_rates(calls, hits, writes, write):
    ratio = cost_ratio(calls, hits, writes, write)
    return f"{hits / calls:7.2%} {float(ratio):7.4f}"


def record_result(title, passed):
    status = "PASS" if passed else "FAIL"
    results[title] = status
    print(f"판정 {status}")


calls = 1_000 if QUICK else 10_000
results = {}

In [ ]:
# Chart language and style
import os
import platform
import sys
from pathlib import Path
from urllib.request import urlopen

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.text import Text

CHART_LANG = os.environ.get('CHART_LANG', 'ko')
CHART_COLORS = ('#2a78d6', '#eb6834', '#1baf7a')
CHART_TEXT = '#0b0b0b'
CHART_MUTED = '#52514e'
CHART_GRID = '#e4e3df'
CHART_BG = '#fcfcfb'
CHART_LIMIT = '#e34948'

in_colab = 'google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ
chart_font = matplotlib.rcParamsDefault['font.family']
if CHART_LANG == 'ko':
    # NanumGothic is licensed under the SIL OFL 1.1.
    nanum = Path('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
    try:
        if not nanum.is_file():
            nanum = Path('fonts/NanumGothic-Regular.ttf')
            if not nanum.is_file():
                nanum.parent.mkdir(exist_ok=True)
                url = 'https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Regular.ttf'
                with urlopen(url, timeout=60) as response:
                    nanum.write_bytes(response.read())
        font_manager.fontManager.addfont(str(nanum))
        chart_font = font_manager.FontProperties(fname=str(nanum)).get_name()
    except Exception as exc:
        CHART_LANG = 'en'
        reason = ' '.join(str(exc).splitlines()) or type(exc).__name__
        print(f'차트 언어 en, 한글 글꼴 설정 실패, {reason}')
    else:
        print(f'차트 언어 ko, 글꼴 {chart_font}')
else:
    CHART_LANG = 'en'

matplotlib.rcParams.update({
    'figure.figsize': (8, 4.5),
    'figure.dpi': 200,
    'savefig.dpi': 200,
    'figure.facecolor': CHART_BG,
    'savefig.facecolor': CHART_BG,
    'axes.facecolor': CHART_BG,
    'axes.edgecolor': CHART_MUTED,
    'axes.labelcolor': CHART_TEXT,
    'axes.titlelocation': 'left',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.axisbelow': True,
    'axes.unicode_minus': False,
    'axes.prop_cycle': matplotlib.rcsetup.cycler(color=CHART_COLORS),
    'axes.grid': True,
    'axes.grid.axis': 'y',
    'grid.color': CHART_GRID,
    'grid.linestyle': '-',
    'grid.linewidth': 0.8,
    'text.color': CHART_TEXT,
    'xtick.color': CHART_MUTED,
    'ytick.color': CHART_MUTED,
    'lines.linewidth': 2,
    'lines.markersize': 6,
    'legend.frameon': False,
    'font.family': chart_font,
})

chart_location = 'Colab' if in_colab else platform.system()
CPU_RUNTIME = f'{chart_location} CPU ({platform.machine()}), Python {platform.python_version()}'


def finish_chart(fig, runtime=CPU_RUNTIME, top=1):
    for text in fig.findobj(Text):
        label = text.get_text()
        if '·' in label:
            text.set_text(label.replace(' ·', ',').replace('·', ','))
    runtime = runtime.replace(' ·', ',').replace('·', ',')
    fig.tight_layout(rect=(0, 0.065, 1, top))
    fig.text(0.02, 0.025, runtime, fontsize=8, color=CHART_MUTED)
    plt.show()

## 1. 기본 읽기 배수에서의 손익분기

캐시를 한 번 쓰고 같은 접두부를 재사용합니다. 일반 입력과 비용이 같거나 낮아지는 최소 히트 수와, 비용이 실제로 낮아지는 최소 히트 수를 구합니다.

In [ ]:
def minimum_hits(write, read, *, strict=False):
    """일반 입력 단가 이하(또는 미만)가 되는 최소 히트 수를 구한다."""
    if write < 1 or (write == 1 and not strict):
        return 0
    if read >= 1:
        return None
    boundary = (write - 1) / (1 - read)
    return floor(boundary) + 1 if strict else ceil(boundary)


def breakeven_experiment(read):
    print(f"읽기 배수 {float(read):g}, 비용 비율 = (W + R·H)/(1 + H)")
    print("   H |     5분 |   1시간")
    for hits in HIT_COUNTS:
        short = cost_ratio(1 + hits, hits, 1, W5, read)
        long = cost_ratio(1 + hits, hits, 1, W60, read)
        print(f"{hits:4d} | {float(short):7.4f} | {float(long):7.4f}")

    print("비용 ≤ 1 ⇔ (1 − R)·H ≥ W − 1")
    passed = True
    for label, _, write in TTLS:
        breakeven = minimum_hits(write, read)
        saving = minimum_hits(write, read, strict=True)
        for threshold, strict in ((breakeven, False), (saving, True)):
            target = "절감" if strict else "손익분기"
            if threshold is None:
                print(f"{label} {target}: 없음")
                passed &= read >= 1 and (write >= 1 if strict else write > 1)
                continue

            ratio = cost_ratio(1 + threshold, threshold, 1, write, read)
            passed &= ratio < 1 if strict else ratio <= 1
            if threshold > 0:
                previous = cost_ratio(threshold, threshold - 1, 1, write, read)
                passed &= previous >= 1 if strict else previous > 1
            print(f"{label} {target}: {threshold}회")
    return passed


record_result("기본 읽기 배수", breakeven_experiment(R))

In [ ]:
L = {
    'ko': {
        'title': '캐시 재사용 횟수별 입력 비용', 'x': '재사용 횟수 H', 'y': '캐시 미사용 대비 비용 비율',
        'minutes': 'TTL 5분', 'hour': 'TTL 1시간', 'source': '읽기 배수 R={read:g}',
    },
    'en': {
        'title': 'Input cost by number of cache reuses', 'x': 'Number of reuses (H)',
        'y': 'Cost relative to uncached input', 'minutes': '5-minute TTL', 'hour': '1-hour TTL',
        'source': 'Cache read multiplier R={read:g}',
    },
}
labels = L[CHART_LANG]
reuse_counts = range(11)
fig, ax = plt.subplots()
for write, name, marker in ((W5, 'minutes', 'o'), (W60, 'hour', 's')):
    ratios = [float(cost_ratio(1 + hits, hits, 1, write, R)) for hits in reuse_counts]
    ax.plot(reuse_counts, ratios, marker=marker, label=labels[name])
ax.set(title=labels['title'], xlabel=labels['x'], ylabel=labels['y'])
ax.set_xticks(reuse_counts)
ax.legend(loc='upper right')
read_note = labels['source'].format(read=float(R))
finish_chart(fig, f'{CPU_RUNTIME} · {read_note}')

## 2. 비교 읽기 배수에서의 손익분기

첫 코드 셀의 `ALT_R`로 읽기 배수만 바꿔 같은 계산을 반복합니다.

In [ ]:
record_result("비교 읽기 배수", breakeven_experiment(ALT_R))

## 3. 포아송 도착에서의 호출 간격

호출 사이 시간을 지수분포에서 뽑습니다. 같은 간격 표본에 대해 읽을 때 TTL을 갱신하는 조건과 쓸 때만 TTL을 설정하는 조건의 히트율과 비용을 비교합니다.

In [ ]:
def poisson_experiment(calls, min_cache_tokens, compare_no_refresh):
    print(f"간격별 {calls:,}회, 첫 호출은 시각 0, 시드 {SEED}")
    print("각 칸은 히트율 / 비용 비율이며 첫 호출도 포함")
    rng = random.Random(SEED)
    gap_samples = [
        (label, [rng.expovariate(1 / mean) for _ in range(calls - 1)])
        for label, mean in MEAN_GAPS
    ]
    passed = True
    refresh_counts = {}
    for refresh in (True, False) if compare_no_refresh else (True,):
        print("읽기 시 TTL 갱신" if refresh else "쓰기 시에만 TTL 설정")
        print("평균 간격 |           5분   |         1시간")
        for label, gaps in gap_samples:
            rates = []
            for _, ttl, write in TTLS:
                hits, writes = simulate(
                    gaps, ttl, min_cache_tokens, min_cache_tokens, refresh
                )
                sliding_hits = sum(gap < ttl for gap in gaps)
                passed &= hits + writes == calls and writes >= 1
                passed &= (hits == sliding_hits if refresh else 0 <= hits <= sliding_hits)
                rates.append(format_rates(calls, hits, writes, write))
                if refresh:
                    refresh_counts[ttl, label] = hits, writes
            print(f"{label:>6} | {rates[0]} | {rates[1]}")
    return passed, refresh_counts

poisson_passed, refresh_counts = poisson_experiment(calls, MIN_CACHE_TOKENS, True)
record_result("포아송 도착의 호출 간격", poisson_passed)

In [ ]:
L = {
    'ko': {
        'title': '포아송 도착의 평균 호출 간격별 입력 비용', 'x': '평균 호출 간격 (초, 로그 눈금)',
        'y': '캐시 미사용 대비 비용 비율', 'minutes': 'TTL 5분', 'hour': 'TTL 1시간',
        'loss': '1.0 초과: 손해', 'source': '읽기 시 TTL 갱신 · 간격당 {calls:,}회 호출',
    },
    'en': {
        'title': 'Input cost by mean gap between Poisson arrivals', 'x': 'Mean time between calls (s, log scale)',
        'y': 'Cost relative to uncached input', 'minutes': '5-minute TTL', 'hour': '1-hour TTL',
        'loss': 'Above 1.0: extra cost', 'source': 'TTL refreshed on read · {calls:,} calls per mean gap',
    },
}
labels = L[CHART_LANG]
mean_seconds = [mean for _, mean in MEAN_GAPS]
fig, ax = plt.subplots()
for (_, ttl, write), name, marker in zip(TTLS, ('minutes', 'hour'), ('o', 's')):
    ratios = []
    for label, _ in MEAN_GAPS:
        hits, writes = refresh_counts[ttl, label]
        ratios.append(float(cost_ratio(calls, hits, writes, write)))
    ax.plot(mean_seconds, ratios, marker=marker, label=labels[name])
ax.axhline(1.0, color=CHART_LIMIT, linestyle='--', linewidth=1.2)
ax.annotate(labels['loss'], (0.02, 1.0), xycoords=('axes fraction', 'data'),
            xytext=(0, 6), textcoords='offset points', va='bottom')
ax.set(title=labels['title'], xlabel=labels['x'], ylabel=labels['y'])
ax.set_xscale('log')
tick_indices = (0, 1, 3, 6, 8, 9)
ticks = [mean_seconds[index] for index in tick_indices]
ax.set_xticks(ticks, [f'{mean:,}' for mean in ticks])
ax.margins(y=0.12)
ax.legend(loc='upper left')
call_note = labels['source'].format(calls=calls)
finish_chart(fig, f'{CPU_RUNTIME} · {call_note}')

## 4. TTL 직전과 직후의 고정 간격

호출 간격을 TTL보다 1초 짧거나 길게 고정합니다. 만료 시점부터는 캐시 미스로 계산합니다.

In [ ]:
def fixed_gap_experiment(calls, min_cache_tokens, compare_no_refresh):
    print(f"조건별 {calls:,}회, 각 칸은 히트율 / 비용 비율")
    heading = "TTL / 간격 |              갱신"
    if compare_no_refresh:
        heading += " |         갱신 없음"
    print(heading)
    passed = True
    for label, ttl, write in TTLS:
        for offset in (1, -1):
            gap = ttl + offset
            gaps = [gap] * (calls - 1)
            hits, writes = simulate(gaps, ttl, min_cache_tokens, min_cache_tokens)
            expected_hits = 0 if offset > 0 else calls - 1
            passed &= hits == expected_hits and writes == calls - expected_hits
            row = f"{label} {offset:+}초 | {format_rates(calls, hits, writes, write)}"
            if compare_no_refresh:
                hits, writes = simulate(
                    gaps, ttl, min_cache_tokens, min_cache_tokens, refresh=False
                )
                calls_per_write = (ttl + gap - 1) // gap
                expected_writes = (calls + calls_per_write - 1) // calls_per_write
                passed &= writes == expected_writes and hits == calls - expected_writes
                row += f" | {format_rates(calls, hits, writes, write)}"
            print(row)
    return passed

record_result("TTL 전후 1초의 고정 간격", fixed_gap_experiment(calls, MIN_CACHE_TOKENS, True))

## 5. 최소 캐시 길이

접두부가 최소 캐시 길이보다 1토큰 짧은 경우와 최소 길이를 충족하는 경우를 비교합니다.

In [ ]:
def minimum_length_experiment(calls, min_cache_tokens):
    prefix_tokens = min_cache_tokens - 1
    gaps = [1] * (calls - 1)
    passed = True
    ratios = []
    for _, ttl, write in TTLS:
        hits, writes = simulate(gaps, ttl, prefix_tokens, min_cache_tokens)
        ratio = cost_ratio(calls, hits, writes, write)
        ratios.append(ratio)
        at_minimum = simulate(gaps, ttl, min_cache_tokens, min_cache_tokens)
        passed &= (hits, writes) == (0, 0) and ratio == 1
        passed &= at_minimum == (calls - 1, 1)
    print(f"{prefix_tokens} < {min_cache_tokens}토큰 → 비용 비율 {ratios[0]} / {ratios[1]} (5분/1시간)")
    return passed

record_result("최소 캐시 길이", minimum_length_experiment(calls, MIN_CACHE_TOKENS))

## 판정 요약

손익분기와 캐시 동작이 계산 조건에 맞는지 확인합니다.

In [ ]:
print("| 실험 | 항목 | 판정 |")
print("| --- | --- | --- |")
for number, (title, status) in enumerate(results.items(), start=1):
    print(f"| {number} | {title} | {status} |")